### Main paper plot 4

2 plots: 4_1 is latepool variations vs one disco. 4_2 is just disco variations.

X-axis: time taken corresponding to the iteration (log scale)

Y-axis: F(S, Q)

Each curve represents the performance of a method.

In [1]:
import torch
import pickle
import os
import numpy as np

In [2]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm

In [3]:
from plot_utils import crop_pdf_with_fitz, crop_pdf_with_pdfcrop, crop_pdf_with_pypdf

In [4]:
%load_ext autoreload
%autoreload 1

%aimport plot_utils

In [ ]:
# Collect data from text files for all datasets
datasets = ['msmarco', 'hotpotqa', 'fever', 'pooled', 'science', 'technology', 'writing']
time_map, max_time_vals = plot_utils.get_time_data(datasets)

In [ ]:
max_time_vals

In [ ]:
# Load score data
score_map = {ds: {} for ds in datasets}
ind_map = {ds: {} for ds in datasets}
methods = ['disco - 1', 'disco - 10', 'disco - 15', 'late poollate pool'late pool
           'late pool - 20']

for ds in datasets:
    for method in methods:
        if method == "gold":
            continue
        inds, scores = plot_utils.get_score_data(ds, method, k=10)
        print(f"Shapes for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}")
        assert inds.shape == scores.shape, f"Shape mismatch for {ds}, {method}: inds: {inds.shape}, scores: {scores.shape}"
        assert inds.shape[0] > 10
        score_map[ds][method] = scores.mean(dim=0).numpy()
        ind_map[ds][method] = inds

In [ ]:
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter

def plot_paper(dataset_name, desired_methods, filename, xticks, yticks, y_label=True, k_cutoff=4, log_scale=False):
    plt.clf()
    fig, ax = plt.subplots(figsize=(8, 6))
    markersize = 10

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
        'lines.markersize': markersize
    })

    exact_greedy_time = np.array(time_map[dataset_name]["exact greedy"] * 10)
    print(exact_greedy_time)

    for method in desired_methods:
        query_time = time_map[dataset_name][method]
        scores = score_map[dataset_name][method].tolist()
        query_times = [query_time * i for i in range(1, len(scores) + 1)]

        # TODO: Do we want only 4 points?
        scores = scores[:k_cutoff]
        query_times = query_times[:k_cutoff]

        efficiency = exact_greedy_time / query_times

        label = plot_utils.method_label_map[method]
        if method == "late pool - 1":
            label = plot_utils.method_label_map[method] + r",\ \textbf{n\textquotesingle = 1}"

        print(f"Method: {method}, Time: {query_time}, Score: {scores}")
        ax.plot(
            # [query_time * i for i in range(1, k_cutoff + 1)],
            efficiency,
            scores,
            label=label,
            color=plot_utils.legend_color_map[plot_utils.method_label_map[method]],
            marker=plot_utils.legend_marker_map[plot_utils.method_label_map[method]],
            linewidth=6,
            markersize=14,
        )

    # ax.set_title(f'{dataset_name}: F(S) vs K', fontsize=20)
    # ax.set_xlabel(r'$\textbf{Retrieval time (sec)}$', fontsize=48)      # Increased axis label size
    ax.set_xlabel(r'$\textbf{Efficiency}\rightarrow$', fontsize=48)      # Increased axis label size
    if y_label:
        ax.set_ylabel(r'$\textbf{Avg}\quad\pmb{F(S_K, Q)}$', fontsize=48) # Increased axis label size

    # Xticks logscale
    # ax.set_xscale('log')
    # Explicitly set tick label sizes
    ax.tick_params(axis='x', labelsize=50)  # Smaller X-axis tick labels
    ax.tick_params(axis='y', labelsize=50)  # Smaller Y-axis tick labels

    # ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    # ax.set_xticks([10, 100, 1000])
    # # Get current x-tick values and format them with LaTeX bold
    # xticks = ax.get_xticks()
    # # Display in 10^i format, and use bold for the numbers
    # powers = [f"10^{int(np.log10(v))}" for v in xticks if v > 0]
    # xticklabels = [fr'$\mathbf{{{p}}}$' for p in powers]
    # ax.set_xticklabels(xticklabels)

    # Get current x-tick values and format them with LaTeX bold
    # xticks = ax.get_xticks()
    # print(ax.get_xticks())
    # xticklabels = []
    # for v in xticks:
    #     if v.is_integer():
    #         xticklabels.append(fr'$\mathbf{{{int(v)}}}$')
    #     else:
    #         xticklabels.append(fr'$\mathbf{{{v}}}$')
    # ax.set_xticklabels(xticklabels)

    # Get current y-tick values and format them with LaTeX bold
    xticks = ax.get_xticks()
    xticklabels = [fr'$\mathbf{{{int(v)}}}$' for v in xticks]
    ax.set_xticklabels(xticklabels)

    # Get current y-tick values and format them with LaTeX bold
    yticks = ax.get_yticks()
    yticklabels = [fr'$\mathbf{{{v}}}$' for v in yticks]
    ax.set_yticklabels(yticklabels)

    ax.xaxis.set_label_coords(0.45, -0.2)

    # ax.legend(fontsize=12)
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(f'./notebooks/plots/{filename}.pdf')
    plt.show()

In [ ]:
def plot_legend_only(desired_methods, filename, ncol=3, auto_crop=True):
    # Create a figure for legend only
    fig, ax = plt.subplots(figsize=(8, 6))

    plt.rcParams.update({
        'text.usetex': True,
        'text.latex.preamble': r'\usepackage{amsmath}',
        'font.family': 'serif',
        'figure.dpi': 300,
    })
    
    for method in desired_methods:
        label = plot_utils.method_label_map[method]
        if method == "late pool
            label = plot_utils.method_label_map[method] + r",\ \textbf{n\textquotesingle = 1}"
        ax.plot(
            [], [],
            label=label,
            color=plot_utils.legend_color_map[plot_utils.method_label_map[method]],
            marker=plot_utils.legend_marker_map[plot_utils.method_label_map[method]],
            linewidth=5,
            markersize=30,
        )

    # Hide the plot area
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Create legend
    legend = ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=ncol,
        fontsize=50,
        frameon=False
    )

    # Save the figure
    output_path = f'./notebooks/plots/{filename}.pdf'
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()
    
    # Auto-crop the PDF if requested
    if auto_crop:
        # Try methods in order of preference
        cropped_path = crop_pdf_with_pdfcrop(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_fitz(output_path)
        if cropped_path is None:
            cropped_path = crop_pdf_with_pypdf(output_path)
        
        if cropped_path:
            # Replace original with cropped version
            os.rename(cropped_path, output_path)
            print(f"PDF automatically cropped: {output_path}")
        else:
            print("Auto-cropping failed, using matplotlib's bbox_inches='tight' only")

In [ ]:
methods_4_1 = ['disco - 1', 'late pool - 1', 'late pool - 10', 'late pool - 15', 'late pool - 20']

In [ ]:
plot_legend_only(methods_4_1, "plot4_1_legend", ncol=3)

In [ ]:
xticks = [10, 25, 40, 55]
yticks = [27, 28]
plot_paper("msmarco", methods_4_1, "msmarco_plot_4_1", xticks, yticks, y_label=True)

In [ ]:
xticks = [15, 30, 45, 60]
yticks = [24, 26]
plot_paper("fever", methods_4_1, "fever_plot_4_1", xticks, yticks, y_label=True)

In [ ]:
xticks = [15, 30, 45, 60]
yticks = [23, 25]
plot_paper("hotpotqa", methods_4_1, "hotpotqa_plot_4_1", xticks, yticks, y_label=False)

In [ ]:
xticks = [10, 20, 30, 40]
yticks = [23, 24, 25]
plot_paper("pooled", methods_4_1, "pooled_plot_4_1", xticks, yticks, y_label=False)

In [ ]:
xticks = [5, 15, 25]
yticks = [22, 23, 24]
plot_paper("science", methods_4_1, "science_plot_4_1", xticks, yticks, y_label=False)

In [ ]:
xticks = [1, 3, 5]
yticks = [23, 24, 25]
plot_paper("writing", methods_4_1, "writing_plot_4_1", xticks, yticks, y_label=False)

In [ ]:
xticks = [3, 7, 11, 15]
yticks = [23, 24, 25]
plot_paper("technology", methods_4_1, "technology_plot_4_1", xticks, yticks, y_label=False)

### 4.2 Bypass Variations

In [ ]:
methods_4_2 = ['disco - 1', 'disco - 10', 'disco - 15']

In [ ]:
plot_legend_only(methods_4_2, "plot4_2_legend", ncol=3)

In [ ]:
xticks = [10, 30, 50, 70]
yticks = [27, 28]
plot_paper("msmarco", methods_4_2, "msmarco_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)

In [ ]:
xticks = [10, 30, 50, 70]
yticks = [24, 26]
plot_paper("fever", methods_4_2, "fever_plot_4_2", xticks, yticks, y_label=True, k_cutoff=5)

In [ ]:
xticks = [10, 25, 40, 55]
yticks = [23, 25]
plot_paper("hotpotqa", methods_4_2, "hotpotqa_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)

In [ ]:
xticks = [10, 20, 30, 40]
yticks = [23, 25]
plot_paper("pooled", methods_4_2, "pooled_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)

In [ ]:
xticks = [5, 10, 15, 20, 25]
yticks = [22, 24]
plot_paper("science", methods_4_2, "science_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)

In [ ]:
xticks = [1, 3, 5, 7, 9]
yticks = [22, 24, 26]
plot_paper("writing", methods_4_2, "writing_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)

In [ ]:
xticks = [3, 7, 11, 15]
yticks = [23, 25, 27]
plot_paper("technology", methods_4_2, "technology_plot_4_2", xticks, yticks, y_label=False, k_cutoff=5)